# Spectral hole burning in Pr:YSO

Worked through linearly. Run the cells in order.



## 1. Config

In [ ]:

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt


# Where the figures appear. "vscode" draws them inline in the notebook;
# "browser" opens each one in a new tab; "notebook" / "notebook_connected"
# for classic Jupyter or JupyterLab. Leaving it as None lets plotly work it
# out for itself, which is usually right.
pio.renderers.default = "vscode"

# Every plot is drawn twice: plotly to look at on screen, matplotlib for
# the write-up. The matplotlib settings live in the next cell rather than
# in a separate style file, so the notebook stands on its own.

params = dict(

    crystal_params = {


        #        +-5/2(g)      +-3/2(g)  +-1/2(g)
        'g': [-(10.19 + 17.30),    -10.19,      0.00],   # MHz

        #        +-1/2(e)  +-3/2(e)     +-5/2(e)
        'e': [        0.00,      4.58,   4.58 + 4.84],   # MHz

        # Relative transition strengths, Nilsson et al. 2004 Table 1
        # and Sharma et al. 2023

        #                      +-1/2(e)  +-3/2(e)  +-5/2(e)
        'dm': np.array([
            [                    0.052,    0.022,    0.926],   #  +-5/2(g)
            [                    0.396,    0.596,    0.008],   #  +-3/2(g)
            [                    0.552,    0.382,    0.066],   #  +-1/2(g)
        ]),
    },

    linewidth_params = {
        'h_width':  0.001,        # homogeneous FWHM (MHz), Lorentzian
        'in_width': 10000.0,      # inhomogeneous FWHM (MHz), Gaussian
    },

    numerical_params = {
        'fmin': -50.0,            # spectrum window, low edge (MHz)
        'fmax':  50.0,            # spectrum window, high edge (MHz)
        'df':    0.001,            # mesh step (MHz)                  #####0.001 takes a very long time
                                  #
                                  # df MUST DIVIDE THE LEVEL ENERGIES ABOVE
                                  # EXACTLY

    },

    burn_params = {
        'pump_passes': 1,         # how many times to repeat the pump list.
 
    },

    output_params = {
        'alpha': 1.0,          # Abs coefficient (1/m) (1/m chosen to correspond with Bloch-Simulator). Set to 1 for normalised Abs
        'norm':  None,            # peak of the unburned spectrum - computed
                                  # just below, every time this cell is run.
    },
)


def settings(params):
    """All the parameters as one flat dict, which is how the functions read them."""
    flat = {}
    for section in params.values():
        flat.update(section)
    return flat


# The normalisation factor sets the OD scale, and it depends on df, h_width,
# in_width and the transitions. After the next cell has run at least once you can just re-run this cell 
# if any of the parameters are changed.
if 'normalisation_factor' in globals():
    params['output_params']['norm'] = normalisation_factor(params)
    print(f"normalisation factor: {params['output_params']['norm']:.6f}")
else:
    print("Run the plot style cell and then the simulator cell below - "
          "the simulator computes the normalisation.")

### Plot style

The matplotlib settings, written out in full so that the notebook does
not need `signature.mplstyle` - or any other file - sitting beside it.

In [ ]:
# ---------------------------------------------------------------------------
# Plot style
#
# Text is Times New Roman. Maths - the Delta in $\Delta$Absorption, sub-
# scripts, and so on - is set in STIX, which is Times drawn to the same
# proportions and comes with matplotlib, so the maths matches the text on
# any machine without anyone having to install a font. To use literal Times
# New Roman glyphs in the maths as well, comment out the 'stix' line and
# uncomment the four beneath it; on a machine without Times New Roman
# installed matplotlib then falls back to its own serif face.
# ---------------------------------------------------------------------------
from cycler import cycler

plt.rcParams.update({

    # --- figure -----------------------------------------------------------
    'figure.figsize':                (8, 5),
    'figure.dpi':                    150,
    'figure.facecolor':              'white',
    'figure.edgecolor':              'white',
    'figure.autolayout':             False,
    'figure.constrained_layout.use': True,

    # --- text -------------------------------------------------------------
    'font.family':      'serif',
    'font.serif':       ['Times New Roman', 'DejaVu Serif'],
    'font.size':        12,
    'text.usetex':      False,

    # --- maths ------------------------------------------------------------
    'mathtext.fontset': 'stix',
    # 'mathtext.fontset': 'custom',
    # 'mathtext.rm':      'Times New Roman',
    # 'mathtext.it':      'Times New Roman:italic',
    # 'mathtext.bf':      'Times New Roman:bold',

    # --- axes -------------------------------------------------------------
    'axes.facecolor':              'white',
    'axes.edgecolor':              '0.15',
    'axes.linewidth':              1.2,
    'axes.grid':                   True,
    'axes.grid.which':             'major',
    'axes.axisbelow':              True,
    'axes.titlesize':              12,
    'axes.titleweight':            'bold',
    'axes.titlepad':               8,
    'axes.labelsize':              12,
    'axes.labelpad':               5,
    'axes.labelweight':            'normal',
    'axes.spines.top':             False,
    'axes.spines.right':           False,
    'axes.formatter.use_mathtext': True,
    'axes.prop_cycle': cycler('color', ['#1f77b4', '#e05c2a', '#2ca02c',
                                        '#9467bd', '#8c564b', '#d62728',
                                        '#17becf', '#bcbd22']),

    # --- grid -------------------------------------------------------------
    'grid.color':     '0.85',
    'grid.linestyle': '-',
    'grid.linewidth': 0.6,
    'grid.alpha':     0.7,

    # --- lines ------------------------------------------------------------
    'lines.linewidth':       2,
    'lines.markersize':      5,
    'lines.markeredgewidth': 0.8,

    # --- ticks (minor ticks on, both pointing inwards) ---------------------
    'xtick.direction':     'in',
    'ytick.direction':     'in',
    'xtick.major.size':    4.5,
    'ytick.major.size':    4.5,
    'xtick.minor.size':    2.5,
    'ytick.minor.size':    2.5,
    'xtick.major.width':   1.1,
    'ytick.major.width':   1.1,
    'xtick.minor.width':   0.8,
    'ytick.minor.width':   0.8,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.labelsize':     10,
    'ytick.labelsize':     10,
    'xtick.bottom':        True,
    'xtick.top':           False,
    'xtick.labelbottom':   True,
    'xtick.labeltop':      False,
    'ytick.left':          True,
    'ytick.right':         False,
    'ytick.labelleft':     True,
    'ytick.labelright':    False,

    # --- legend -----------------------------------------------------------
    'legend.fontsize':     12,
    'legend.frameon':      True,
    'legend.framealpha':   1.0,
    'legend.facecolor':    'white',
    'legend.edgecolor':    'black',
    'legend.fancybox':     True,
    'legend.shadow':       True,
    'legend.borderpad':    0.5,
    'legend.labelspacing': 0.4,
    'legend.handlelength': 1.8,
    'legend.loc':          'best',

    # --- images (the heatmap) ---------------------------------------------
    'image.cmap':          'inferno',
    'image.interpolation': 'nearest',
    'image.aspect':        'auto',
    'image.origin':        'lower',

    # --- saving -----------------------------------------------------------
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.05,
    'savefig.facecolor':  'white',
    'savefig.edgecolor':  'white',
    'savefig.format':     'png',
})

## 2. SHB simulator

In [ ]:
def to_steps(values, df):
    """
    MHz to whole df steps. df must divide every frequency in the simulation
    exactly
    """
    steps = np.asarray(values, dtype=float) / df
    if np.any(np.abs(steps - np.round(steps)) > 1e-6):
        raise ValueError(f"not a whole number of df = {df:g} steps - see "
                         f"the note in the config")
    return np.round(steps).astype(np.int64)
def build_transitions(params):
    """
    The table of transition frequencies, carried in whole df steps.

    Returns
        g, e         sorted energies (MHz)
        trans        [Ng, Ne], trans[k, m] = e[m] - g[k]   (MHz)
        trans_steps  the same table in whole df steps      (int)
    """
    p = settings(params)
    df = p['df']
    g = np.asarray(p['g'], dtype=float)
    e = np.asarray(p['e'], dtype=float)

    g_steps = to_steps(g, df)
    e_steps = to_steps(e, df)

    trans_steps = e_steps[None, :] - g_steps[:, None]
    return g, e, trans_steps * df, trans_steps


def make_axes(params, trans_steps, fmin=None, fmax=None):
    """
    The spectrum axis f and the atomic detuning axis det.

    Returns f, det, and det0 - the bin index of det[0], which is what turns a
    frequency in steps into a column of rho.
    """
    p = settings(params)
    df = p['df']
    fmin = p['fmin'] if fmin is None else fmin
    fmax = p['fmax'] if fmax is None else fmax

    f0 = int(to_steps(fmin, df))
    f1 = int(to_steps(fmax, df))
    pad = int(np.abs(trans_steps).max())

    f = df * np.arange(f0, f1 + 1)
    det = df * np.arange(f0 - pad, f1 + pad + 1)
    return f, det, f0 - pad


def initial_populations(g, det):
    """Equal population in every ground state, for every ion class."""
    return np.ones((len(g), len(det))) / len(g)


def burn_parallel(rho, pump_freqs, trans_steps, det, det0, params,
                  pump_passes=None):
    """
    PARALLEL burn - every pump frequency acts at once.

    All the frequencies are present together, so they all act on the SAME
    populations: the ions go up together and decay back down together. The
    order of pump_freqs therefore makes no difference to the answer.

    rho goes in, a new rho comes out; the input is not modified.
    """
    p = settings(params)
    pump_passes = p['pump_passes'] if pump_passes is None else pump_passes
    df, dm = p['df'], p['dm']
    n_g, n_e = trans_steps.shape
    rho = rho.copy()

    # The pump frequencies as whole bins. This is where an off-mesh comb is
    # caught, before any of it is burned.
    pump_steps = to_steps(pump_freqs, df)

    for _ in range(pump_passes):

        # hits[k, d]        how many addressed transitions take ions out of
        #                   ground state k in ion class d
        # landing[j, k, d]  the branchings of those same transitions, summed -
        #                   so landing/hits is where the ions actually go
        # I still need to check this so that it is 100% correct
        hits = np.zeros_like(rho)
        landing = np.zeros((n_g, n_g, len(det)))

        for pump_step in pump_steps:
            for k in range(n_g):
                for m in range(n_e):

                    # The ion class whose (k, m) transition sits exactly on
                    # this pump frequency. Integer subtraction - the bin is
                    # computed, not rounded to.
                    d = int(pump_step - trans_steps[k, m] - det0)
                    if not (0 <= d < len(det)):
                        continue

                    hits[k, d] += 1
                    landing[:, k, d] += dm[:, m]

        # The ions leave. A class addressed three times over still only has
        # one population to give, so this is a mask, not a count.
        excited = rho * (hits > 0)

        # And come back down, sharing equally between the branchings of
        # whichever transitions took them.
        share = np.divide(landing, hits,
                          out=np.zeros_like(landing), where=hits > 0)
        rho = rho - excited + (share * excited).sum(axis=1)

    return rho

def burn_series(rho, pump_freqs, trans_steps, det, det0, params,
                pump_passes=None):
    """
    SERIES burn - one pump frequency after another.
    """
    p = settings(params)
    pump_passes = p['pump_passes'] if pump_passes is None else pump_passes
    df, dm = p['df'], p['dm']
    n_g, n_e = trans_steps.shape
    rho = rho.copy()

    pump_steps = to_steps(pump_freqs, df)

    for _ in range(pump_passes):
        for pump_step in pump_steps:
            for k in range(n_g):
                for m in range(n_e):

                    d = int(pump_step - trans_steps[k, m] - det0)
                    if not (0 <= d < len(det)):
                        continue

                    removed = rho[k, d]
                    rho[k, d] = 0.0
                    rho[:, d] += removed * dm[:, m]

    return rho

def spectrum(rho, trans, det, f, params, norm=None, alpha=None):
    """
    Absorption spectrum from the populations.

    norm and alpha default to config. the spectrum is divided by norm and
    multiplied by alpha, which puts the unburned crystal at alpha.
    """
    p = settings(params)
    norm = (p['norm'] if p['norm'] is not None else 1.0) if norm is None else norm
    alpha = p['alpha'] if alpha is None else alpha
    h_width, in_width, dm = p['h_width'], p['in_width'], p['dm']

    spec = np.zeros(len(f))
    half = h_width / 2.0
    n_g, n_e = trans.shape

    for d in range(len(det)):
        gauss = np.exp(-4 * np.log(2) * det[d]**2 / in_width**2)
        for k in range(n_g):
            pop = rho[k, d]
            if pop == 0.0:
                continue
            for m in range(n_e):
                spec += (dm[k, m] * pop * half**2
                         / ((trans[k, m] + det[d] - f)**2 + half**2)
                         * gauss)

    return (spec / norm) * alpha


def run_burn(pump_freqs, params, burn=burn_parallel,
             pump_passes=None, fmin=None, fmax=None):
    """
    The whole thing: set up, burn, take the spectrum.

    Pass burn=burn_series for a series burn instead of a parallel one.

    Returns a dictionary with
        f, spec       the spectrum and its frequency axis
        rho, rho0     populations after and before the burn
        det           the atomic detuning axis (the x axis for rho)
        trans         the transition table (MHz)
        pump_freqs    what was burned
    """
    g, e, trans, trans_steps = build_transitions(params)
    f, det, det0 = make_axes(params, trans_steps, fmin, fmax)
    rho0 = initial_populations(g, det)
    rho = burn(rho0, pump_freqs, trans_steps, det, det0, params, pump_passes)
    spec = spectrum(rho, trans, det, f, params)

    return {'f': f, 'spec': spec, 'rho': rho, 'rho0': rho0,
            'det': det, 'trans': trans, 'trans_steps': trans_steps,
            'pump_freqs': np.asarray(pump_freqs)}


def normalisation_factor(params):
    """
    Peak absorption of the unburned crystal, measured over a 100 MHz window at
    the centre of the inhomogeneous line.

    Dividing a spectrum by this and multiplying by alpha puts the unburned
    crystal at alpha, with holes below it and antiholes above.

    It has to be computed rather than written down
    """
    g, e, trans, trans_steps = build_transitions(params)
    f, det, det0 = make_axes(params, trans_steps, fmin=-50.0, fmax=50.0)
    rho = initial_populations(g, det)
    spec = spectrum(rho, trans, det, f, params, norm=1.0, alpha=1.0)
    return float(spec.max())


# Takes a few seconds. From now on, re-running the config cell refreshes it.
params['output_params']['norm'] = normalisation_factor(params)
print(f"normalisation factor: {params['output_params']['norm']:.6f}")


---
## 3. A single hole

The simplest burn: one pump frequency, burned ten times over.

In [ ]:
# --- THE BURN --------------------------------------------------------------
# Run this once. The two plot cells below read `out`, so they can be re-run
# and edited as often as you like without burning again.

def single_hole(centre=0.0):
    """One pump frequency, at `centre` MHz."""
    return np.array([centre])


centre      = 0.0    # MHz, where the hole is burned
pump_passes = 10     # how many times the pump is repeated

pump_freqs = single_hole(centre=centre)

out = run_burn(pump_freqs, params, burn=burn_parallel, pump_passes=pump_passes)

print(f"{len(pump_freqs)} pump frequency, {pump_passes} passes")
print(f"OD {out['spec'].min():.0f} to {out['spec'].max():.0f}")

#### Plots

Both cells below read the burn above, so they can be re-run and edited
without burning again.

In [ ]:
# --- PLOTLY, to look at on screen ------------------------------------------
# Reads `out` from the burn cell above. Edit the settings here; nothing in
# this cell changes the matplotlib style used for the write-up figures.

spec_title  = "A single hole burned at 0 MHz, 10 passes"
spec_colour = "#C05621"
spec_width  = 0.9
spec_xlim   = None          # e.g. (-40, 40), or None for the full window
spec_ylim   = None

pops_title  = "Ion classes addressed by the burn"
pops_width  = 1.1
pops_xlim   = None
pops_ylim   = None


# --- PLOT 1: spectrum ---
fig = go.Figure()
fig.add_hline(y=settings(params)['alpha'], line_dash="dash",
              line_color="#94A3B8", line_width=1)
fig.add_trace(go.Scatter(x=out['f'], y=out['spec'], mode="lines",
                         line=dict(width=spec_width, color=spec_colour),
                         name="spectrum"))
fig.update_layout(title=spec_title,
                  xaxis_title="Frequency (MHz)",
                  yaxis_title="Absorption (OD)",
                  xaxis_range=spec_xlim, yaxis_range=spec_ylim,
                  template="plotly_white")
fig.show()

# --- PLOT 2: addressed ion classes ---
fig = go.Figure()
for k in range(out['rho'].shape[0]):
    fig.add_trace(go.Scatter(x=out['det'], y=out['rho'][k], mode="lines",
                             line=dict(width=pops_width),
                             name=f"ground state {k}"))
fig.update_layout(title=pops_title,
                  xaxis_title="Atomic detuning (MHz)",
                  yaxis_title="Population",
                  xaxis_range=pops_xlim, yaxis_range=pops_ylim,
                  template="plotly_white")
fig.show()

In [ ]:
# --- MATPLOTLIB, for the write-up ------------------------------------------
# Reads `out` from the burn cell above. Everything set here overrides the
# notebook's plot style for these two figures only - change a value, re-run
# this cell, and nothing else in the notebook is affected.

save_png = True                      # write the two .png files

# spectrum
spec_figsize = (12, 3)
spec_xlim    = (-40, 40)             # zooms in on the burned hole at 0 MHz
spec_ylim    = (0.4, 1.2)            # normalised range, with a little padding
spec_colour  = 'black'
spec_lw      = 1
spec_title   = 'A Single Hole Burned At 0 MHz'
spec_legend  = False
spec_file    = 'single_hole_spectrum.png'

# populations
pops_figsize = (8, 5)
pops_xlim    = (-40, 5)              # set by the width of the laser interactions
pops_ylim    = (-0.05, 1.05)         # population is a fraction, 0 to 1
pops_lw      = 1
pops_title   = 'Ion classes addressed by the burn'
pops_legend  = True
pops_file    = 'single_hole_populations.png'


p = settings(params)

# --- PLOT 1: optical spectrum ---
fig, ax = plt.subplots(figsize=spec_figsize)
ax.axhline(p['alpha'], color='0.5', lw=0.8, ls='--', label='unburned')
ax.plot(out['f'], out['spec'], color=spec_colour, lw=spec_lw, label='spectrum')
ax.set_xlim(spec_xlim)
ax.set_ylim(spec_ylim)
ax.set_xlabel(r'Frequency (MHz)')
ax.set_ylabel(r'Absorption (Normalised)')
ax.set_title(spec_title)
if spec_legend:
    ax.legend()

# Saved before it is shown: on the inline backend a figure can be cleared by
# show, and a blank png written instead.
if save_png:
    fig.savefig(spec_file)
plt.show()

# --- PLOT 2: ion populations ---
fig, ax = plt.subplots(figsize=pops_figsize)
for k in range(out['rho'].shape[0]):
    ax.plot(out['det'], out['rho'][k], lw=pops_lw, label=f"ground state {k}")
ax.set_xlim(pops_xlim)
ax.set_ylim(pops_ylim)
ax.set_xlabel(r'Atomic detuning (MHz)')
ax.set_ylabel(r'Population')
ax.set_title(pops_title)
if pops_legend:
    ax.legend()

if save_png:
    fig.savefig(pops_file)
plt.show()

---
## 4. Scanning tooth spacing and burn width percentage

Sweeps tooth spacing against burn width, scoring each spectrum by its
ΔOD over a window at the centre.

The scan runs on the config's window and mesh, and steps the tooth spacing.

The cost of a fixed window is the wide end: a 10 MHz tooth spacing has to fit
inside the same +-20 MHz that a 0.1 MHz spacing uses, which leaves only five
teeth. 


### The scan

In [ ]:


import heapq
import time
import numpy as np
from tqdm import tqdm


def comb(spacing_steps, width_steps, params):
    """
    Teeth every spacing_steps bins across the spectrum window, each tooth
    width_steps bins wide.

    """
    p = settings(params)
    df = p['df']
    f0 = int(to_steps(p['fmin'], df))
    f1 = int(to_steps(p['fmax'], df))

    n_teeth = (f1 - f0) // spacing_steps + 1
    starts = f0 + spacing_steps * np.arange(n_teeth)
    return df * (starts[:, None] + np.arange(width_steps)[None, :]).ravel()


def run_scan(spacing_min, spacing_max, spacing_step,
             n_widths, target_window, top_n, pump_passes):
    """
    Sweep tooth spacing and burn width, scoring each spectrum by its
    delta OD over a window at the centre.

    The spectrum window and mesh come from the config - fmin, fmax and df cant be changed here. 
    Change them in cell 1 and re-run it to re-normalise

    spacing_min/max  range of tooth spacings to try (MHz)
    spacing_step     increment between them (MHz). All three are converted to
                     whole df steps and must divide by df exactly.
    n_widths         burn widths per spacing - the tooth is divided into this
                     many, so widths run spacing/n
    target_window    judge each spectrum over this width at the centre (MHz).
                     Defaults to 4 x spacing_max, so the judging window always
                     holds several teeth however wide the spacing gets. Be carefull with using this
    top_n            how many of the best spectra to keep for plotting

    Returns a dictionary:
        tag        short name for this scan, used on filenames and titles
        results    (spacing, width, delta_od) for every point, in MHz
        best       the top_n records, largest delta OD first
        params     the parameters this scan ran with (the config)
        n_widths, target_window, pump_passes
    """
    p = settings(params)
    df = p['df']
    tag = f"{spacing_min:g}-{spacing_max:g}MHz"

    # Judge over a window wide enough to hold several teeth at the widest
    # spacing, rather than a fixed width that would hold one tooth or none.
    if target_window is None:
        target_window = spacing_max*4


    s_min = int(to_steps(spacing_min, df))
    s_max = int(to_steps(spacing_max, df))
    s_inc = int(to_steps(spacing_step, df))
    spacing_grid = np.arange(s_min, s_max + 1, s_inc)

    grid = [(int(s), max(1, int(n * s // n_widths)))
            for s in spacing_grid
            for n in range(1, n_widths + 1)]

    # one spectrum,  for the timer
    print(f"[{tag}]  {len(spacing_grid)} spacings, {len(grid)} spectra   "
          f"window {p['fmin']:g} to {p['fmax']:g} MHz, df {df:g}, judged over "
          f"{target_window:g} MHz")

    _t0 = time.time()
    run_burn(comb(*grid[len(grid) // 2], params), params,
             burn=burn_parallel, pump_passes=pump_passes)
    _total = (time.time() - _t0) * len(grid)
    print(f"    about " + (f"{_total:.0f} seconds" if _total < 90
                             else f"{_total / 60:.0f} minutes"))


    # run the scan
    best, results = [], []
    for n, (spacing_steps, width_steps) in enumerate(
            tqdm(grid, desc=f"scan {tag}", unit="spec")):

        out = run_burn(comb(spacing_steps, width_steps, params), params,
                       burn=burn_parallel, pump_passes=pump_passes)
        f, spec = out['f'], out['spec']

        centre = f[len(f) // 2]
        lo = np.searchsorted(f, centre - target_window / 2)
        hi = np.searchsorted(f, centre + target_window / 2)
        delta_od = spec[lo:hi].max() - spec[lo:hi].min()

        tooth_spacing = spacing_steps * df
        results.append((tooth_spacing, width_steps * df, delta_od))

        # Only records the top spectrums as it slows down the plotly plot
        record = (delta_od, n, spec, tooth_spacing, width_steps, f)
        if len(best) < top_n:
            heapq.heappush(best, record)
        elif delta_od > best[0][0]:
            heapq.heappushpop(best, record)

    return {'tag': tag,
            'results': results,
            'best': sorted(best, key=lambda r: r[0], reverse=True),
            'params': params,
            'n_widths': n_widths,
            'target_window': target_window,
            'pump_passes': pump_passes}


### The three plots

In [ ]:
# Matplotlib versions for the write-up, plotly versions for looking at on
# screen. The matplotlib ones follow signature.mplstyle - the same style
# sheet as mbe_simulation.ipynb - so figures from both notebooks match.
import matplotlib.ticker as mtick

#-------------------------------------------------------------------------------------------------------------------
def plot_heatmap_mp(scan, save=True):
    """ΔOD over tooth spacing and burn width fraction."""
    n_widths = scan['n_widths']
    res = scan['results']

    spacing_vals = np.array(sorted({s for s, w, d in res}))

    def fraction(s, w):
        return round(w / s * n_widths) / n_widths

    frac_vals = np.array(sorted({fraction(s, w) for s, w, d in res}))

    z = np.full((len(frac_vals), len(spacing_vals)), np.nan)
    for s, w, d in res:
        col = int(np.searchsorted(spacing_vals, s))
        row = int(np.searchsorted(frac_vals, fraction(s, w)))
        z[row, col] = d

    fig, ax = plt.subplots(figsize=(12, 3))

    # shading='nearest' centres each pixel on its spacing and fraction. Any
    # combination the scan did not visit stays NaN and is drawn blank.
    #
    # viridis (blue -> green -> yellow) is named here rather than left to the
    # style sheet, whose inferno is kept for anything else. It matches the
    # plotly heatmap below, so the two versions of this figure read the same.
    mesh = ax.pcolormesh(spacing_vals, frac_vals, z, shading='nearest',
                         cmap='viridis')
    fig.colorbar(mesh, ax=ax, label=r'$\Delta$Absorption')

    ax.set_title(f"AFC $\\Delta$Absorption Over the Central "
                 f"{scan['target_window']:g} MHz")
    ax.set_xlabel(r'OFC Pulse spacing (MHz)')
    ax.set_ylabel(r'Pulse Width Percentage of Spacing')
    # Full 0-100% axis. The scan only burns 10% to 90% of a tooth, so there is
    # a blank band top and bottom - that is the range of the data, not a gap.
    ax.set_ylim(0.05, 0.95)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))

    # Saved before it is shown: on the inline backend a figure can be cleared
    # by show, and a blank png written instead.
    if save:
        fig.savefig(f"DeltaOD_heatmap_{scan['tag']}.png")
    plt.show()


def plot_max_dod_mp(scan, save=True):
    """The best ΔOD achievable at each tooth spacing."""
    res = scan['results']

    best_per = {}
    for s, w, d in res:
        if s not in best_per or d > best_per[s][0]:
            best_per[s] = (d, w)

    spacing_vals = np.array(sorted(best_per))
    max_dod = np.array([best_per[s][0] for s in spacing_vals])
    best_width = np.array([best_per[s][1] for s in spacing_vals])

    fig, ax = plt.subplots(figsize=(12, 3))

    ax.plot(spacing_vals, max_dod, color='C0', lw=0.8,
            label=r'Best $\Delta$Absorption')

    # Ring the best point rather than marking every one - at 0.05 MHz steps
    # there are far too many spacings for markers to read.
    i = int(np.argmax(max_dod))
    ax.plot(spacing_vals[i], max_dod[i], linestyle='none', marker='o',
            markersize=11, markerfacecolor='none', markeredgecolor='C1',
            markeredgewidth=1.8, label=f"Best: pulse spacing {spacing_vals[i]:.3f} MHz, pulse width 1.175 MHz")

    ax.set_title(f"Best $\\Delta$Absorption at Each Pulse Spacing")
    ax.set_xlabel(r'OFC Pulse Spacing (MHz)')
    ax.set_ylabel(r'Maximum $\Delta$Absorption')
    ax.legend(fontsize=8, loc='lower right')

    if save:
        fig.savefig(f"DeltaOD_vs_toothspacing_{scan['tag']}.png")
    plt.show()

    print(f"[{scan['tag']}] best: \u0394Absorption {max_dod[i]:.0f} at "
          f"{spacing_vals[i]:.2f} MHz, width {best_width[i]:.3f} MHz ")
#---------------------------------------------------------------------------------------------------------



def plot_top_spectra(scan, save=True):
    """The top spectrums overlaid, for visual inspection."""
    p = settings(scan['params'])
    df = p['df']

    fig = go.Figure()
    fig.add_hline(y=p['alpha'], line_dash="dash", line_color="#94A3B8",
                  line_width=1)
    for delta_od, _, spec, tooth_spacing, pulses, f in scan['best']:
        width = pulses * df
        fig.add_trace(go.Scatter(
            x=f, y=spec, mode="lines", line=dict(width=0.7),
            name=f"ΔOD {delta_od:.2f} | spacing {tooth_spacing:.2f} MHz, "
                 f"width {width:.3f} MHz ({width / tooth_spacing:.0%})"))

    fig.update_layout(
        title=f"Top {len(scan['best'])} spectra by ΔOD  ({scan['tag']})"
              f"<br>central {scan['target_window']:g} MHz, window "
              f"±{p['fmax']:g} MHz, df {df:g}, "
              f"{scan['pump_passes']} pass parallel",
        xaxis_title="Frequency (MHz)", yaxis_title="Normalised Absorption",
        template="plotly_white")
    fig.show()
    if save:
        fig.write_html(f"TopSpectra_{scan['tag']}.html")


def plot_heatmap(scan, save=True):
    """ΔOD over tooth spacing and burn width fraction."""
    n_widths = scan['n_widths']
    res = scan['results']

    spacing_vals = np.array(sorted({s for s, w, d in res}))

    def fraction(s, w):
        return round(w / s * n_widths) / n_widths

    frac_vals = np.array(sorted({fraction(s, w) for s, w, d in res}))

    z = np.full((len(frac_vals), len(spacing_vals)), np.nan)
    width_mhz = np.full_like(z, np.nan)
    for s, w, d in res:
        col = int(np.searchsorted(spacing_vals, s))
        row = int(np.searchsorted(frac_vals, fraction(s, w)))
        z[row, col] = d
        width_mhz[row, col] = w

    fig = go.Figure(data=go.Heatmap(
        x=spacing_vals, y=frac_vals, z=z, customdata=width_mhz,
        colorscale="Viridis", colorbar=dict(title="ΔOD"),
        hovertemplate="spacing %{x:.2f} MHz<br>%{y:.0%} of the tooth burned "
                      "(%{customdata:.3f} MHz)<br>ΔOD %{z:.0f}<extra></extra>"))

    fig.update_layout(
        title=f"ΔOD over the central {scan['target_window']:g} MHz  "
              f"({scan['tag']})<br>{len(res)} burns, "
              f"{scan['pump_passes']} pass parallel",
        xaxis_title="Tooth spacing (MHz)",
        yaxis_title="Burn width fraction (pulse width / tooth spacing)",
        yaxis=dict(tickformat=".0%"),
        template="plotly_white")
    fig.show()
    if save:
        fig.write_html(f"DeltaOD_heatmap_{scan['tag']}.html")


def plot_max_dod(scan, save=True):
    """The best ΔOD achievable at each tooth spacing."""
    res = scan['results']

    best_per = {}
    for s, w, d in res:
        if s not in best_per or d > best_per[s][0]:
            best_per[s] = (d, w)

    spacing_vals = np.array(sorted(best_per))
    max_dod = np.array([best_per[s][0] for s in spacing_vals])
    best_width = np.array([best_per[s][1] for s in spacing_vals])

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=spacing_vals, y=max_dod, mode="lines+markers",
        line=dict(width=1.4, color="#C05621"), marker=dict(size=5),
        name="best ΔOD",
        customdata=np.stack([best_width, best_width / spacing_vals], axis=-1),
        hovertemplate="spacing %{x:.2f} MHz<br>ΔOD %{y:.0f}"
                      "<br>best width %{customdata[0]:.3f} MHz "
                      "(%{customdata[1]:.0%})<extra></extra>"))

    i = int(np.argmax(max_dod))
    fig.add_trace(go.Scatter(
        x=[spacing_vals[i]], y=[max_dod[i]], mode="markers",
        marker=dict(size=11, color="rgba(0,0,0,0)",
                    line=dict(color="#2C7A7B", width=2)),
        name=f"best: {spacing_vals[i]:.2f} MHz", hoverinfo="skip"))

    fig.update_layout(
        title=f"Best ΔOD at each tooth spacing  ({scan['tag']})"
              f"<br>central {scan['target_window']:g} MHz, "
              f"{scan['pump_passes']} pass parallel",
        xaxis_title="Tooth spacing (MHz)", yaxis_title="Maximum ΔOD",
        template="plotly_white")
    fig.show()
    if save:
        fig.write_html(f"DeltaOD_vs_toothspacing_{scan['tag']}.html")

    print(f"[{scan['tag']}] best: ΔOD {max_dod[i]:.0f} at "
          f"{spacing_vals[i]:.2f} MHz, width {best_width[i]:.3f} MHz "
          f"({best_width[i] / spacing_vals[i]:.0%} filled)")

### Run the scan



In [ ]:
# One scan. Window and mesh come from the config, so the only thing set here
# is the range of tooth spacings and the increment between them 

#scan = run_scan(spacing_min=0.2, spacing_max=5.0, spacing_step=0.01, n_widths=10, target_window=20, top_n=50, pump_passes=10)


In [ ]:
import pickle
with open("results_20260824_174011.pkl", "rb") as f:
    scan = pickle.load(f)

### Show plots


In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
#Absorption plots for dissertation
plot_heatmap_mp(scan)
plot_max_dod_mp(scan)
#------------------------------

plot_top_spectra(scan)   # I know that this one is awkward but it is really useful to be able to see the best spectrums
                         # It doesnt work perfectly but by clicking on the legend at the side
                         # 3 or 4 times (two double clicks) it eventually isolates a spectrum. 
                         # I found that I needed to keep trying this until it worked
#plot_heatmap(scan)
#plot_max_dod(scan)


---
## 5. One chosen comb

Pick a comb by hand, run it, save it, plot it.

Set `tooth_spacing` and `burn_width` in the cell below. They can come off the
scan's heatmap, or from a physical argument that has nothing to do with it
the cell does not need a scan to have been run. Again be careful with the mesh.



In [ ]:
# --- THE BURN --------------------------------------------------------------
# Run this once. The two plot cells below read `f`, `spec` and `tag`, so they
# can be re-run and edited as often as you like without burning again.

tooth_spacing = 2.35      # MHz, gap between neighbouring teeth
burn_width    = 1.175     # MHz, how much of each tooth is burned
pump_passes   = 10      # how many times the whole comb is repeated

save_array    = True    # write the .npy


# --- pump frequencies -------------------------------------------------
def comb_at(tooth_spacing, burn_width, params):
    """
    A comb across the whole config window: teeth every `tooth_spacing` MHz,
    each tooth `burn_width` MHz wide.

    Returns
        freqs           the pump frequencies (MHz), exactly on the mesh
        actual_spacing  the spacing, on the mesh (MHz)
        actual_width    the width, on the mesh (MHz)
        n_teeth         how many teeth fitted in the window
    """
    p = settings(params)
    df = p['df']

    # to_steps rather than round: an off-mesh spacing or width stops here
    spacing_steps = int(to_steps(tooth_spacing, df))
    width_steps   = int(to_steps(burn_width, df))

    f0 = int(to_steps(p['fmin'], df))
    f1 = int(to_steps(p['fmax'], df))

    n_teeth = (f1 - f0) // spacing_steps + 1
    starts = f0 + spacing_steps * np.arange(n_teeth)
    freqs = df * (starts[:, None] + np.arange(width_steps)[None, :]).ravel()

    return freqs, spacing_steps * df, width_steps * df, n_teeth


pump_freqs, actual_spacing, actual_width, n_teeth = comb_at(
    tooth_spacing, burn_width, params)

p = settings(params)

# run the burn
out = run_burn(pump_freqs, params, burn=burn_parallel,
               pump_passes=pump_passes)

f, spec = out['f'], out['spec']
delta_od = spec.max() - spec.min()

print(f"{n_teeth} teeth, {len(pump_freqs)} pump frequencies, "
      f"{pump_passes} pass parallel")
print(f"spacing {actual_spacing:g} MHz, width {actual_width:g} MHz "
      f"({actual_width / actual_spacing:.0%} of the tooth filled)")
print(f"Absorption {spec.min():.0f} to {spec.max():.0f}   (DAbsorption {delta_od:.0f}, "
      f"unburned {p['alpha']:.0f})")

# save an array in whatever filepath you are working in
# Row 0 is the frequency axis in MHz, row 1 the absorption in OD, so the
# it saves a 2D array and a 1D one that is just the spectrum without a frequency axis

tag = f"df{p['df']:g}_{actual_spacing:g}MHz_{actual_width:g}MHz"

if save_array:
    np.save(f"{tag}.npy", np.vstack([f, spec]))
    np.save(f"{tag}spec.npy", spec)
    print(f"saved {tag}.npy   shape (2, {len(f)})")

#### Plots

Both cells below read the burn above, so they can be re-run and edited
without burning again.

In [ ]:
# --- PLOTLY, to look at on screen ------------------------------------------
# Reads `f`, `spec` and `tag` from the burn cell above.

save_html = False        # write a standalone plotly page

title  = f"Comb: {actual_spacing:g} MHz spacing, {actual_width:g} MHz width "
colour = "#6D2E0F"
width  = 0.9
xlim   = None            # e.g. (-20, 20), or None for the full window
ylim   = None


fig = go.Figure()
fig.add_hline(y=p['alpha'], line_dash="dash", line_color="#94A3B8",
              line_width=1, annotation_text="unburned",
              annotation_position="bottom right")
fig.add_trace(go.Scatter(
    x=f, y=spec, mode="lines", line=dict(width=width, color=colour),
    name="spectrum",
    hovertemplate="%{x:.3f} MHz<br>Abs %{y:.0f}<extra></extra>"))

fig.update_layout(
    title=title,
    xaxis_title="Frequency (MHz)",
    yaxis_title="Absorption (Normalised)",
    xaxis_range=xlim, yaxis_range=ylim,
    template="plotly_white")
fig.show()

if save_html:
    fig.write_html(f"{tag}.html")
    print(f"saved {tag}.html")

In [ ]:
# --- MATPLOTLIB, for the write-up ------------------------------------------
# Reads `f`, `spec` and `tag` from the burn cell above. Everything set here
# overrides the notebook's plot style for this figure only.

save_png = False         # write the .png

figsize = (8, 4)
xlim    = (-10,10)           # e.g. (-20, 20), or None for the full window
ylim    = (0,3)
colour  = 'black'
lw      = 0.8
title   = f"AFC Created With {actual_spacing:g} MHz OFC Pulse Spacing, {actual_width:g} MHz Pulse Width"
legend  = False


fig, ax = plt.subplots(figsize=figsize)
ax.axhline(p['alpha'], color='0.5', lw=0.6, ls='--', label='unburned')
ax.plot(f, spec, color=colour, lw=lw, label='spectrum')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel(r'Frequency (MHz)')
ax.set_ylabel(r'Absorption (Normalised)')
ax.set_title(title)
if legend:
    ax.legend()

# Saved before it is shown: on the inline backend a figure can be cleared by
# show, and a blank png written instead.
if save_png:
    fig.savefig(f"{tag}.png")
    print(f"saved {tag}.png")
plt.show()